<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP8.2A/MP8.2A_thermal_cycling.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP8.2A · Thermal Cycling with Natural Convection

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'heatsink.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08-heatsink-cooling/")
PROJECT = (['mp82a.py', 'MP8.2A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP8.2A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'heatsink', 'mp82a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import heatsink as hs
import mp82a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP8.2A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

A power module - the IGBTs or MOSFETs of a motor drive, an inverter or a charger - is rarely loaded steadily. A lift, a crane or a test bench runs it in **duty cycles**: on for a while, off for a while, over and over. Every cycle heats the module and lets it cool again, and every swing of temperature strains the solder layers and the bond wires, because silicon, copper and aluminium expand by different amounts. Fatigue from these swings is the most common way a power module wears out, and the number of cycles a module survives falls steeply with the **size of the swing** and with the **peak temperature** reached.

So the quantity an engineer wants is not one temperature but a history: **the temperature of the module, cycle by cycle**, its peak and its trough, and how both change with the module's power. A lifetime study repeats this for many mission profiles and many powers, so the model is run many times. That is where a fast model pays.

This mini project computes that history for the heat sink of Ex08, under a train of ten pulses, **for any module power between 100 and 200 W**, with one network.

## A2 · The heat sink, and what is new against Ex08

The heat sink is Ex08's: an extruded aluminium sink with a 100 mm wide, 10 mm thick base plate and ten straight fins, cooled by a fan in an enclosure at 40 °C. The module sits on the middle 30 mm of the base. The cell below draws it and prints its data.

**What is new.** Ex08 cools the fins with one constant coefficient, $h = 40$ W/(m²K), the fan's forced air. In a real enclosure the air the fins warm also rises by its own buoyancy - **natural convection** - and the hotter the fins, the stronger it is. The coefficient on the fins becomes

$$h(\Delta T) = h_\mathrm{fan} + S \cdot C_\mathrm{nat}\left(\frac{\Delta T}{L_\mathrm{fin}}\right)^{1/4}$$

where $\Delta T$ is the rise of the fin root above the air, $L_\mathrm{fin} = 40$ mm the fin length, and $C_\mathrm{nat} = 1.42$ W/(m$^{7/4}$K$^{5/4}$) the laminar natural-convection coefficient of a vertical plate in air (Incropera & DeWitt, ch. 9; **assumed** here, from memory). $S = 1$ is the strength of the change; at $S = 0$ the problem is exactly Ex08's.

The fins' efficiency depends on $h$, so the **effective coefficient** the finned face puts on the base plate, $h_\mathrm{eff}$, is Ex08's fin calculation evaluated with $h(\Delta T)$. It rises from 322 W/(m²K) with the sink at the air's temperature to about 370 W/(m²K) at a 20 K rise.

In [ ]:
# The heat sink of Ex08, and its data
fig, ax = plt.subplots(figsize=(8, 3.6))
hs.draw_heatsink(ax)                                     # Ex08's drawing of the sink, the module and the domain
plt.show()

rows = [("base plate", f"{hs.W_BASE*1e3:.0f} mm wide, {hs.T_BASE*1e3:.0f} mm thick, {hs.DEPTH*1e3:.0f} mm deep"),
        ("aluminium 6061", f"k = {hs.K_AL:.0f} W/(m·K), ρc_p = {hs.RHO_CP/1e6:.2f} MJ/(m³K)"),
        ("fins", f"{hs.N_FINS} fins, {hs.T_FIN*1e3:.0f} × {hs.L_FIN*1e3:.0f} mm, pitch {hs.PITCH*1e3:.0f} mm"),
        ("air", f"{hs.T_AIR:.0f} °C, fan h = {hs.H_AIR:.0f} W/(m²K)"),
        ("module", f"footprint {hs.MODULE[0]*1e3:.0f}–{hs.MODULE[1]*1e3:.0f} mm, power P = {mp.P_MIN:.0f}–{mp.P_MAX:.0f} W"),
        ("duty cycle", f"on {mp.T_ON:.0f} s every {mp.PERIOD:.0f} s, {mp.CYCLES} cycles, {mp.T_END:.0f} s in all")]
for k, v in rows:
    print(f"{k:>15}  {v}")

**What you should see.** The sink in cross-section, the module on top, the domain - the base plate - outlined; and the table of its data. Every value is Ex08's except the power range and the duty cycle.

In [ ]:
# The coefficient on the fins, with and without natural convection
dT = np.linspace(0, 45, 200)                             # K, the fin-root rise above the air
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(dT, mp.h_air(dT), label="fan + natural convection")
ax[0].plot(dT, mp.h_air(dT, s_nat=0.0), "--", label="fan only (Ex08)")
ax[0].set_xlabel("fin-root rise ΔT [K]"); ax[0].set_ylabel("h on the fins [W/(m²K)]"); ax[0].legend()
ax[1].plot(dT, mp.h_eff(dT), label="fan + natural convection")
ax[1].plot(dT, mp.h_eff(dT, s_nat=0.0), "--", label="fan only (Ex08)")
ax[1].set_xlabel("fin-root rise ΔT [K]"); ax[1].set_ylabel("$h_{eff}$ on the base [W/(m²K)]"); ax[1].legend()
plt.tight_layout(); plt.show()
for d in (0, 10, 20, 40):
    print(f"ΔT = {d:2d} K:  h = {float(mp.h_air(d)):5.1f} W/(m²K),  h_eff = {float(mp.h_eff(d)):5.1f} W/(m²K)")

**What you should see.** The fan alone gives a flat line; natural convection adds a part that grows with the fourth root of the rise - steeply at first, then slowly. On the base, $h_\mathrm{eff}$ goes from 322 W/(m²K) at no rise to 370 at 20 K and 379 at 40 K: about 15 % more cooling when the sink is hot, which is enough to lower the peak of the tenth cycle by almost 2 K at 150 W.

## A3 · The domain, the equation and its conditions

The fins are collapsed into $h_\mathrm{eff}$, as in Ex08, so the domain is the **base plate in cross-section**: $0 < x < W$ across the base, $0 < z < t_b$ through its thickness ($z = 0$ the fin side, $z = t_b$ the module side), per metre of depth. The unknown is the rise above the air, $\theta(x, z, t) = T - T_\mathrm{air}$, which depends also on the module's power $P$.

$$\rho c_p \frac{\partial \theta}{\partial t} = k\left(\frac{\partial^2 \theta}{\partial x^2} + \frac{\partial^2 \theta}{\partial z^2}\right) \quad \text{in the base plate}$$

| where | condition | why it holds |
|---|---|---|
| module face, on the footprint ($z = t_b$) | $k\,\partial\theta/\partial z = \dfrac{P}{P_0}\,q''\,s(t)$ | the module's losses enter the base through its baseplate; $q'' = 50$ kW/m² at $P_0 = 150$ W, and $s(t)$ is 1 while the module is on |
| module face, off the footprint | $\partial\theta/\partial z = 0$ | still air above the base plate; its heat loss is small against the fins' |
| fin side ($z = 0$) | $k\,\partial\theta/\partial z = h_\mathrm{eff}(\theta)\,\theta$ | the fins, collapsed into one coefficient that now depends on the local rise: **nonlinear** |
| ends ($x = 0$ and $x = W$) | $\partial\theta/\partial x = 0$ | the two 10 mm ends lose little against 0.8 m² of fins |
| start ($t = 0$) | $\theta = 0$ | the sink has stood switched off at the air's temperature |

The equation is **parabolic**: heat diffuses, and the field at any instant depends only on what came before. Two time scales matter. Through the 10 mm thickness heat spreads in $t_b^2/\alpha \approx 1.5$ s, so the fin side follows the module within seconds. Across the 100 mm base it takes $W^2/\alpha \approx 145$ s, longer than one cycle, so the heat spread sideways builds up over several cycles: the history settles into a repeating pattern only after five to eight cycles.

In [ ]:
# The duty cycle s(t): on for 20 s every 60 s, ten times
t = np.linspace(0, mp.T_END, 6001)
fig, ax = plt.subplots(figsize=(11, 2.2))
ax.fill_between(t, mp.train_on(t).astype(float), step="pre", alpha=0.6)
ax.set_xlabel("t [s]"); ax.set_ylabel("s(t)"); ax.set_yticks([0, 1]); ax.set_xlim(0, mp.T_END)
plt.tight_layout(); plt.show()
print("the module switches", len(mp.SWITCHES), "times:", mp.SWITCHES[:6], "...")

**What you should see.** Ten blocks, 20 s wide, one a minute: 20 switches in all. At every switch the flux jumps, so the temperature's rate of change jumps too: the history the network must represent has **twenty kinks**. Keep that in mind for part D.

## A4 · Why there is no exact solution

Ex08's base plate has an exact solution because every condition is **linear** in $\theta$: the field is a sum of cosines across the base times known functions of $z$ and $t$, and a train of pulses is a sum of steps. With $h_\mathrm{eff}(\theta)$ the fin-side condition multiplies the unknown by a function of itself. A sum of two solutions is then no longer a solution, separation of variables fails, and no series or closed form exists. That is the normal case in engineering: the nonlinearity is mild, but it is enough.

The reference and the ground truth are therefore **numerical**: finite differences in space and Crank-Nicolson in time (part C). They are checked in three ways: on grids each twice as fine, by the energy balance, and at $S = 0$, where they must reproduce Ex08's exact series.

## A5 · What is asked

| quantity | at | required |
|---|---|---|
| the rise of the module's centre, $\theta(W/2, t_b, t)$, over the 600 s | the test powers 115, 165 and 195 W, which you do not train on | every cycle's **peak and trough within 0.3 K** of the ground truth |
| the field across the base plate at 600 s | the same powers | within **0.5 K** everywhere |
| the time to answer a new power online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline, over the whole range of powers. Its online answer - the ten peaks and troughs for a power it has not seen - is what is timed against the reference model, which must march the 600 s again for every new power. The training time is reported apart, as the offline cost.

The trough of a cycle is the rise at the end of that cycle, just before the module switches on again. 0.3 K is the agreed engineering accuracy: about six times the noise of the thermocouple of Ex08, and small against the 13 K swing.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

The network is a function of four inputs and returns one number:

$$(x,\ z,\ t,\ P)\ \longmapsto\ \theta(x, z, t; P)$$

the rise above the air at a point of the base plate, at an instant, for a module power. The power is an **input**, like the coordinates: one training covers every power from 100 to 200 W, and a new power is answered by evaluating the network, not by training again.

Networks train best when their inputs and output are of order one. The cell below prints the scales of this problem, which Ex08 used too.

In [ ]:
# The scales: inputs and output of order one
print(f"X = x / W,     W   = {hs.W_BASE*1e3:.0f} mm")
print(f"Z = z / t_b,   t_b = {hs.T_BASE*1e3:.0f} mm")
print(f"θ̂ = θ / Θ,     Θ   = q_mean / h_eff(0) = {hs.THETA:.2f} K   (the fin-side rise of a 1-D hand calculation at 150 W)")
print(f"diffusion time through the thickness  t_b²/α = {hs.T_BASE**2/hs.ALPHA:.2f} s")
print(f"aspect ratio ε = t_b / W = {hs.EPS:.2f};   Biot number at no rise  Bi = h_eff t_b / k = {hs.BI:.4f}")
print(f"window {mp.T_END:.0f} s; power {mp.P_MIN:.0f}-{mp.P_MAX:.0f} W")

## B2 · The physics the network is trained with

In the scaled variables the heat equation and its conditions read

$$\frac{t_b^2}{\alpha}\,\frac{\partial \hat\theta}{\partial t} = \varepsilon^2\,\frac{\partial^2 \hat\theta}{\partial X^2} + \frac{\partial^2 \hat\theta}{\partial Z^2}$$

$$\frac{\partial \hat\theta}{\partial Z} = Bi\; g(X)\,\frac{P}{P_0}\,s(t) \ \ \text{at } Z = 1, \qquad
\frac{\partial \hat\theta}{\partial Z} = \frac{h_\mathrm{eff}(\Theta\hat\theta)\,t_b}{k}\,\hat\theta \ \ \text{at } Z = 0, \qquad
\frac{\partial \hat\theta}{\partial X} = 0 \ \ \text{at } X = 0, 1, \qquad \hat\theta = 0 \ \ \text{at } t = 0$$

with $g(X) = q''/q_\mathrm{mean}$ on the footprint (3.33) and 0 elsewhere. Each line becomes a term of the loss: the squared residual of the equation at points inside the plate, and of each condition at points on its face.

| loss term | where it is evaluated | what it enforces |
|---|---|---|
| $\mathcal{L}_\mathrm{PDE}$ | interior points $(X, Z, t, P)$ | the heat equation: conduction and storage |
| $\mathcal{L}_\mathrm{module}$ | points on $Z = 1$ | the module's heat enters on the footprint, nothing elsewhere |
| $\mathcal{L}_\mathrm{fins}$ | points on $Z = 0$ | the nonlinear fin condition, with $h_\mathrm{eff}$ at the network's own rise |
| $\mathcal{L}_\mathrm{ends}$ | points on $X = 0, 1$ | insulated ends |
| $\mathcal{L}_\mathrm{start}$ | points at $t = 0$ | the sink starts at the air's temperature |

**What can be built in rather than trained.** Ex08 showed that a condition built into the network's form holds exactly and leaves the training less to do. Here: the insulated ends hold for any weights if the network sees $X$ only through $\cos(\pi X)$, whose slope is zero at both ends; the start holds if the output is written so that it vanishes at $t = 0$. Whether and how you build these in is one of your choices in part D.

**The energy balance, at every instant.** Integrated over the plate, the heat equation says that the heat stored changes at the rate the module puts heat in minus the rate the fins take it out:

$$\frac{t_b^2}{\alpha}\,\frac{d\langle\hat\theta\rangle}{dt} = Bi\,\frac{P}{P_0}\,s(t) - \left\langle \frac{h_\mathrm{eff}\,t_b}{k}\,\hat\theta \right\rangle_{Z=0}$$

where $\langle\cdot\rangle$ is the mean over the plate (left) or over the fin side (right). It follows from the equation and the two face conditions, so a network that satisfied them exactly would satisfy it too. **But no network satisfies them exactly, and over 600 s that matters**: a residual of one part in a thousand in the equation, integrated over a hundred seconds, becomes a drift of the stored heat of several kelvin. Ex08 met the same problem in its steady sink, and its cure was this balance as a loss term. In time it is needed more, not less. Notebook 03 gives it as an optional term, `balance_loss`; whether and how much to weight it is your choice.

In [ ]:
# What the network must represent: the centre's rise at 150 W, from the reference model
ref = mp.reference(150.0)                                # Crank-Nicolson on the engineering grid (part C)
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(ref["t"], ref["centre"], lw=1)
for s in mp.SWITCHES:
    ax.axvline(s, color="grey", lw=0.4, alpha=0.5)
ax.set_xlabel("t [s]"); ax.set_ylabel("θ at the module's centre [K]"); ax.set_xlim(0, mp.T_END)
plt.tight_layout(); plt.show()
print("peaks  :", np.round(ref["peaks"], 2))
print("troughs:", np.round(ref["troughs"], 2))

**What you should see.** A saw-tooth that climbs for about six cycles and then repeats: peaks 16.2, 20.6, 22.4 ... 23.6 K and troughs 6.0, 8.4, 9.4 ... 10.0 K at 150 W, from the reference model (part C shows it is within 0.2 K of the ground truth). The slope changes abruptly at every grey line, a switch. A network that is smooth in $t$ - a plain multilayer perceptron of $t$ - rounds those corners, and the rounding is largest exactly at the peaks it is scored on. Ways around it that you may consider in part D: give the network inputs that carry the switching (functions of $t$ built from $s(t)$), train cycle by cycle, or build a simpler model of the history into the network and let it learn the correction.

## B3 · The training points

A physics-informed network needs no solution to train on: it needs **points** at which to evaluate the residuals. `mp82a.py` provides samplers for the three sets. Two choices are made in them, each a parameter you may change:

* **in time** (`near`), a quarter of the instants fall in the few seconds after a switch (exponentially distributed, mean 4 s), where the field changes fastest; the rest are uniform over the 600 s;
* **on the module face** (`edges`), a tenth of the points fall within 2.5 mm of the footprint's two edges, where the flux steps from zero to 50 kW/m².

Both concentrations have a cost worth knowing. A smooth network cannot follow a step in the flux exactly, so points right at the footprint's edges and right after a switch keep a residual that no training removes. Put too many there and that residual dominates the loss and hides everything else; put none and the network has no reason to be sharp where the field is. The defaults are a compromise; a uniform face (`edges=0`) is a reasonable choice too.

The power is drawn uniformly from 100 to 200 W. The three **test powers, 115, 165 and 195 W**, are inside the range, but no point is placed at them on purpose: the network is scored where it was not fitted.

In [ ]:
# The three sets of training points
X, Z, t, Pw = mp.sample_interior(4000)                    # inside the plate
Xt, tt, Pt = mp.sample_top(1500)                          # on the module face
Xb, tb, Pb = mp.sample_bottom(1500)                       # on the fin side
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].scatter(X, Z, s=1, alpha=0.4, label="interior")
ax[0].scatter(Xt, np.ones_like(Xt), s=2, label="module face")
ax[0].scatter(Xb, np.zeros_like(Xb), s=2, label="fin side")
ax[0].set_xlabel("X = x/W"); ax[0].set_ylabel("Z = z/t_b"); ax[0].legend(fontsize=7, loc="center right")
ax[1].hist(t, bins=300); ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("points"); ax[1].set_xlim(0, mp.T_END)
ax[2].hist(Pw, bins=40); ax[2].set_xlabel("P [W]"); ax[2].set_ylabel("points")
for p in mp.P_TEST:
    ax[2].axvline(p, color="red", lw=1)
plt.tight_layout(); plt.show()

**What you should see.** Left, the plate with its points, slightly denser on the module face round the footprint's edges at $X$ = 0.35 and 0.65. Middle, twenty spikes, one after each switch, over a uniform floor. Right, a flat distribution of powers with the three test powers in red.

## B4 · Measured data, if your design wants some

The physics alone determines the answer, so no measurement is needed. Some designs nevertheless train faster or more robustly with a few data points added to the loss - a **hybrid** network. For that, `mp.thermocouple(P)` returns the record of Ex08's thermocouple, glued to the module face at $x$ = 80 mm, every 2 s with 0.05 K of noise, computed from the ground truth. If you use it, use it at powers other than the test powers, and say in the report what it changed.

In [ ]:
# A thermocouple record at 130 W, with its noise
t_tc, th_tc = mp.thermocouple(130.0)
fig, ax = plt.subplots(figsize=(11, 2.8))
ax.plot(t_tc, th_tc, ".", ms=2)
ax.set_xlabel("t [s]"); ax.set_ylabel("θ at x = 80 mm [K]"); ax.set_xlim(0, mp.T_END)
plt.tight_layout(); plt.show()
print(f"{len(t_tc)} readings, noise {hs.NOISE_K} K")

**What you should see.** 300 readings: the same saw-tooth as the centre's, smaller (the thermocouple is 15 mm beyond the footprint's edge) and later, with the noise visible as a slight scatter.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the solver an engineer would run: finite differences on an engineering grid | the same scheme on a grid fine enough to be ten to a hundred times more exact than anything scored |
| grid | 11 × 3 nodes, a step of 4 s | 161 × 33 nodes, a step of 0.125 s |
| its job | to be **timed** against the network: the online cost of answering a new power | to **score** the network and the reference |

Both march the same discrete equations, written in `mp82a.March`. The rest of this notebook explains the scheme, builds confidence in the ground truth, and measures the reference model's accuracy and time - the bar part D has to clear.

## C2 · The scheme, step by step

1. **Nodes.** A grid of $n_x \times n_z$ nodes covers the base plate, edges included.
2. **Conduction.** At every node the second derivatives are central differences of its neighbours, $(\theta_{i+1} - 2\theta_i + \theta_{i-1})/h^2$.
3. **Conditions by ghost nodes.** A condition on a slope is met by imagining a node one step outside the plate and choosing its value so that the central difference of the slope is what the condition asks. At the insulated ends the ghost mirrors its neighbour. At the module face it carries the flux. At the fin side it carries $h_\mathrm{eff}(\theta)\,\theta$, which depends on the node's own value.
4. **Time: Crank-Nicolson.** Each step averages the right-hand side over the start and the end of the step, which is second-order accurate and stable for any step. It needs one sparse linear solve per step, with a matrix factorised once.
5. **The nonlinear fin side: chord iterations.** The fin term is evaluated at the end of the step, where the value is not yet known. Each step therefore solves three times, each time with the fin term evaluated at the last estimate, using the matrix factorised once with the fin side linearised at a 20 K rise. Three iterations converge to rounding; ten change nothing.
6. **After a switch: Rannacher's start.** When the flux jumps, Crank-Nicolson carries the jump forward as an undamped oscillation and loses its second order. The two steps after every switch are therefore taken as backward-Euler half steps, which damp it. Without them the error falls only by a factor of 2.3 per refinement instead of 4.

`mp.March(nx, nz, dt).run(P)` returns the times, the centre's rise at every step, and the fields at chosen instants.

In [ ]:
# The ground truth: its convergence study at 150 W, from the truth file
G = np.load("MP8.2A_truth.npz")
print("grid (nx × nz, dt)        peak of cycle 10   trough of cycle 10")
for g, p, q in zip(G["conv_grids"], G["conv_peak10"], G["conv_trough10"]):
    print(f"{int(g[0]):4d} × {int(g[1]):3d}, {g[2]:.4f} s      {p:10.4f} K       {q:10.4f} K")
d = np.abs(np.diff(G["conv_peak10"]))
print("change between grids:", np.round(d, 4), "K;  ratios", np.round(d[:-1] / d[1:], 1))
print(f"Richardson extrapolation: {float(G['conv_peak10_extrapolated']):.4f} K")
print(f"at S = 0, the march against Ex08's exact series: {float(G['zero_strength_error_K']):.4f} K at most")

**What you should see.** Four grids, each twice as fine in $x$, $z$ and $t$. The change of the tenth peak between grids falls by about four each time - the second order of the scheme. The ground truth's grid (161 × 33) is within 0.001 K of the extrapolated value, three hundred times finer than the 0.3 K asked. At $S = 0$ the same march reproduces Ex08's exact series to better than 0.001 K: the code is right where the answer is known.

In [ ]:
# The energy balance over the window, on three grids
for g in [(11, 3, 4.0), (41, 9, 0.5), (161, 33, 0.125)]:
    m = mp.March(*g); m.run(150.0)
    b = m.balance
    print(f"{g[0]:4d} × {g[1]:3d}: heat in {b['heat_in']:8.1f} J = through the fins {b['heat_out']:8.1f} J "
          f"+ stored {b['stored']:7.1f} J;  off by {abs(b['residual'])/b['heat_in']:.1e} of the heat in")

**What you should see.** 30 000 J in over the ten pulses; about 27 580 J through the fins and 2 420 J still stored in the base at 600 s; the balance closes to $10^{-3}$ on the coarse grid and $10^{-6}$ on the ground truth's.

## C3 · The reference model: accuracy and time

The reference grid was chosen as the coarsest that meets the agreed 0.3 K on every peak and trough at the test powers. The cell below scores it against the ground truth and times it. It runs on the CPU of the Colab runtime: a small sparse march gains nothing from a GPU, and an engineer would not run it on one.

In [ ]:
# The reference model against the ground truth, at the three test powers
worst = 0.0
for P in mp.P_TEST:
    r = mp.reference(P)
    key = f"{int(P)}W"
    e = max(np.abs(r["peaks"] - G["peaks_" + key]).max(), np.abs(r["troughs"] - G["troughs_" + key]).max())
    worst = max(worst, e)
    print(f"P = {P:.0f} W: peak of cycle 10 {r['peaks'][-1]:.2f} K (truth {G['peaks_' + key][-1]:.2f}),"
          f" worst peak or trough off by {e:.2f} K")
t_ref = middle(lambda: mp.reference(165.0))              # the online cost of a new power
print(f"\nreference model: worst {worst:.2f} K, {t_ref*1e3:.1f} ms per new power")
print(f"the network must answer a new power in under {t_ref*1e3/10:.2f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "worst_ref_K": worst}           # kept for part E

**What you should see.** At every test power the reference model's peaks and troughs are within about 0.21 K of the ground truth, inside the 0.3 K agreed. One run takes some tens of milliseconds (26 to 55 ms on the authors' CPUs), almost all of it the 150 steps of the march. Its time is kept in `REF` for part E, which compares the network against it on this same runtime.

In [ ]:
# The ground truth's history at the three test powers, the target of part D
fig, ax = plt.subplots(figsize=(11, 3.4))
for P in mp.P_TEST:
    ax.plot(G["t"], G[f"centre_{int(P)}W"], lw=0.8, label=f"{P:.0f} W")
ax.set_xlabel("t [s]"); ax.set_ylabel("θ at the module's centre [K]"); ax.set_xlim(0, mp.T_END); ax.legend()
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 2.4))
im = ax.imshow(G["field_165W"], origin="lower", aspect="auto", extent=[0, hs.W_BASE*1e3, 0, hs.T_BASE*1e3])
ax.set_xlabel("x [mm]"); ax.set_ylabel("z [mm]"); plt.colorbar(im, label="θ at 600 s, 165 W [K]")
plt.tight_layout(); plt.show()

**What you should see.** Three saw-tooth histories, higher with the power but not in proportion to it - natural convection cools a hotter sink better. Below, the field at the end of the window at 165 W: hottest under the module, cooler towards the ends, the fin side a few kelvin below the module side. These are what part D's network must reproduce.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else in this notebook is given, so that your month goes into the deep learning.

**The contract.** Your network is a PyTorch module called `net`, used as `net(X, Z, t, P)`:

| argument | meaning | shape |
|---|---|---|
| `X`, `Z` | $x/W$ and $z/t_b$, in 0 to 1 | `(n, 1)` |
| `t` | time in seconds, 0 to 600 | `(n, 1)` |
| `P` | the module's power in watts, 100 to 200 | `(n, 1)` |
| returns | the scaled rise $\hat\theta = \theta/\Theta$, with $\Theta$ = 46.57 K | `(n, 1)` |

Whatever you compute inside - features of $t$, built-in conditions, several sub-networks - stays inside `net`. The loss below and the evaluation at the end only call `net(X, Z, t, P)`.

## D2 · The loss, given

The function `losses(net, seed)` below draws fresh training points with the samplers of part B and returns the three residual terms of part B's table: the heat equation inside, the module face, and the fin side. The ends and the start are not in it: build them into your network (section B2), or add their terms yourself. `balance_loss(net, seed)` is the energy balance at 64 random instants, computed with Gauss quadrature over the plate (section B2): optional, and in our experience decisive over a 600 s window.

Two details worth reading:

* **Every residual is divided by the Biot number**, $Bi = 0.0193$. All three are of that order here - the plate is thin and conducts well, so its temperature varies across the thickness by about $Bi\,\hat\theta$ - and left unscaled they would be a hundred times smaller than a network's usual error, and the optimiser would not see them.
* **The fin coefficient is evaluated at the network's own rise and held constant in the gradient** (`detach`). That is what the reference model's chord iterations do: the coefficient follows the solution, but the optimiser is not asked to differentiate through $\Delta T^{1/4}$, whose slope is infinite at $\Delta T = 0$.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a):
    '''A NumPy array as an (n, 1) tensor on the device.'''
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def d(u, x):
    '''du/dx by automatic differentiation, keeping the graph for a second derivative.'''
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

TB = hs.T_BASE ** 2 / hs.ALPHA                           # s, the diffusion time through the thickness, 1.46 s

def losses(net, seed, n_in=1500, n_face=800):
    # inside the plate: the heat equation, t_b²/α ∂θ̂/∂t = ε² ∂²θ̂/∂X² + ∂²θ̂/∂Z²
    X, Z, t, P = map(T, mp.sample_interior(n_in, seed))
    for v in (X, Z, t):
        v.requires_grad_(True)                           # autograd differentiates with respect to these
    th = net(X, Z, t, P)
    r_pde = TB * d(th, t) - (hs.EPS ** 2 * d(d(th, X), X) + d(d(th, Z), Z))
    # the module face, Z = 1: ∂θ̂/∂Z = Bi g(X) (P/P0) s(t)
    Xt, tt, Pt = mp.sample_top(n_face, seed + 1)
    Zt = torch.ones(len(Xt), 1, device=DEVICE, requires_grad=True)
    top = net(T(Xt), Zt, T(tt), T(Pt))
    r_top = d(top, Zt) - hs.BI * T(hs.footprint(Xt) * mp.train_on(tt) * Pt / mp.P0)
    # the fin side, Z = 0: ∂θ̂/∂Z = h_eff(θ) t_b / k · θ̂, the coefficient at the network's own rise
    Xb, tb, Pb = mp.sample_bottom(n_face, seed + 2)
    Zb = torch.zeros(len(Xb), 1, device=DEVICE, requires_grad=True)
    bot = net(T(Xb), Zb, T(tb), T(Pb))
    bi = T(mp.h_eff(bot.detach().cpu().numpy().ravel() * hs.THETA) * hs.T_BASE / hs.K_AL)
    r_fin = d(bot, Zb) - bi * bot
    return {"pde": ((r_pde / hs.BI) ** 2).mean(),     # each residual divided by Bi: all three are of its order
            "module": ((r_top / hs.BI) ** 2).mean(),
            "fins": ((r_fin / hs.BI) ** 2).mean()}

# the energy balance at random instants (optional; section B2):
#   t_b²/α d<θ̂>/dt = Bi (P/P0) s(t) - <h_eff t_b/k θ̂> at Z = 0
_gx, _wx = np.polynomial.legendre.leggauss(32)           # Gauss points across the base ...
_gz, _wz = np.polynomial.legendre.leggauss(8)            # ... and through the thickness
GX, WX = T((_gx + 1) / 2).ravel(), T(_wx / 2).ravel()    # on (0, 1); the weights sum to one, so a sum is a mean
GZ, WZ = T((_gz + 1) / 2).ravel(), T(_wz / 2).ravel()

def balance_loss(net, seed, n_t=64):
    rng = np.random.default_rng(seed)
    tq = T(mp.sample_times(n_t, rng, 0.25)).requires_grad_(True)       # instants
    Pq = T(rng.uniform(mp.P_MIN, mp.P_MAX, n_t))                       # and powers
    nx, nz = len(GX), len(GZ)
    X = GX.repeat_interleave(nz).repeat(n_t).reshape(-1, 1)            # every quadrature point at every instant
    Z = GZ.repeat(nx).repeat(n_t).reshape(-1, 1)
    w = (WX[:, None] * WZ[None, :]).reshape(-1).repeat(n_t).reshape(-1, 1)
    th = net(X, Z, tq.repeat_interleave(nx * nz, 0), Pq.repeat_interleave(nx * nz, 0))
    mean = (w * th).reshape(n_t, -1).sum(1, keepdim=True)              # <θ̂> over the plate at each instant
    rate = d(mean, tq)                                                 # its time derivative, by autograd
    Xf = GX.repeat(n_t).reshape(-1, 1)
    thf = net(Xf, torch.zeros_like(Xf), tq.repeat_interleave(nx, 0), Pq.repeat_interleave(nx, 0))
    bi = T(mp.h_eff(thf.detach().cpu().numpy().ravel() * hs.THETA) * hs.T_BASE / hs.K_AL)
    out = (WX.repeat(n_t).reshape(-1, 1) * bi * thf).reshape(n_t, -1).sum(1, keepdim=True)
    s = T(mp.train_on(tq.detach().cpu().numpy().ravel()))
    r = TB * rate - (hs.BI * s * Pq / mp.P0 - out)
    return ((r / hs.BI) ** 2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The four inputs, scaled to order one; or features built from them - functions of $X$ whose slope vanishes at the ends, functions of $t$ that carry the duty cycle's switching (section B2);
* **what is built in.** The insulated ends, the start at the air's temperature, the scaling of the rise with the power;
* **its size.** Layers, neurons, activation. Remember why Part 2 uses tanh, not ReLU.

Two things the problem tells you, from parts A and B. The history has twenty kinks, one at each switch. And the module's flux steps from zero to 50 kW/m² at the footprint's edges: across the base the field needs more detail than one smooth function of $X$ gives.

In [ ]:
# TODO 1 - your network. net(X, Z, t, P) -> θ̂, every argument an (n, 1) tensor (see the table above)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers and any trainable numbers

    def forward(self, X, Z, t, P):
        ...                                              # return θ̂, shape (n, 1)

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses` (and, if you choose, `balance_loss`). Decide the optimiser or optimisers, the number of steps, the learning rate and its schedule, the weight of each term, how many points and whether they are drawn afresh each step. Keep the history of each term for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost.

The course's recipe (Adam, then L-BFGS) is a good start. Check the score as you go: `quick_score()` below gives the worst peak or trough error at the three test powers in a second, from the ground truth.

In [ ]:
# A quick score, given: the worst peak or trough error (K) at the test powers
G = np.load("MP8.2A_truth.npz")
def quick_score():
    tq = np.arange(0.0, mp.T_END + 1e-9, 0.5); worst = 0.0
    for P in mp.P_TEST:
        with torch.no_grad():
            th = net(T(np.full(tq.size, 0.5)), T(np.ones(tq.size)), T(tq), T(np.full(tq.size, P))).cpu().numpy().ravel() * hs.THETA
        p, q = mp.cycle_extremes(tq, th)
        worst = max(worst, np.abs(p - G[f"peaks_{int(P)}W"]).max(), np.abs(q - G[f"troughs_{int(P)}W"]).max())
    return worst

In [ ]:
# TODO 2 - your training
history = {"pde": [], "module": [], "fins": [], "balance": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst peak or trough error {quick_score():.3f} K")

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the centre's history at one test power against the ground truth, with its error. Save each figure you want in the report as `MP8.2A_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`, which part E scores:

* the centre's rise every 0.5 s at the test powers 115, 165 and 195 W, at 90 and 210 W outside the range, and at eleven powers from 100 to 200 W;
* the field at 600 s at 165 W on the ground truth's grid;
* the energy balance of your network at 165 W: heat in, heat through the fins, heat stored at 600 s;
* the **online time** of a new power on this runtime - the ten peaks and troughs at 165 W, from one call of `net` - and the same for 10 and 100 powers at once, against the reference model run 1, 10 and 100 times on the same runtime.

Set `TRAIN_SECONDS` to the offline training time you measured; it is reported beside the online time.

In [ ]:
# The evaluation, given ------------------------------------------------------------
import json
net.eval()
tq = np.arange(0.0, mp.T_END + 1e-9, 0.5)                # s, the instants of the history
def centre_history(powers):
    # the centre's rise (K) at every instant, for several powers in one call of the network
    powers = np.atleast_1d(powers)
    n = len(tq)
    X = torch.full((n * len(powers), 1), 0.5, device=DEVICE)
    Z = torch.ones_like(X)
    t = T(np.tile(tq, len(powers)))
    P = T(np.repeat(powers, n))
    with torch.no_grad():
        return (net(X, Z, t, P) * hs.THETA).reshape(len(powers), n).cpu().numpy()

def extremes(hist):
    return np.array([mp.cycle_extremes(tq, h) for h in hist])     # (powers, 2, cycles)

powers = {"test": np.array(mp.P_TEST), "outside": np.array([90.0, 210.0]), "sweep": np.linspace(100, 200, 11)}
out = {f"centre_{k}": centre_history(v) for k, v in powers.items()}
out.update({f"powers_{k}": v for k, v in powers.items()}, t=tq)

# the field at 600 s at 165 W, on the ground truth's grid
G = np.load("MP8.2A_truth.npz")
XX, ZZ = np.meshgrid(G["x"] / hs.W_BASE, G["z"] / hs.T_BASE)
with torch.no_grad():
    out["field_165W"] = (net(T(XX), T(ZZ), T(np.full(XX.size, mp.T_END)), T(np.full(XX.size, 165.0)))
                         * hs.THETA).reshape(XX.shape).cpu().numpy()

# the network's energy balance at 165 W
xs = np.linspace(0, 1, 201); ts = np.arange(0.25, mp.T_END, 0.5)
XB, TT = np.meshgrid(xs, ts)
with torch.no_grad():
    fin_side = (net(T(XB), T(np.zeros(XB.size)), T(TT), T(np.full(XB.size, 165.0))) * hs.THETA).cpu().numpy().reshape(XB.shape)
    end_field = (net(T(XX), T(ZZ), T(np.full(XX.size, mp.T_END)), T(np.full(XX.size, 165.0))) * hs.THETA).cpu().numpy()
q_fins = hs.DEPTH * np.trapezoid(mp.h_eff(fin_side) * fin_side, xs * hs.W_BASE, axis=1)    # W through the fins
heat_out = float(np.sum(q_fins) * 0.5)
stored = float(hs.RHO_CP * hs.DEPTH * np.trapezoid(np.trapezoid(end_field.reshape(XX.shape), G["x"], axis=1), G["z"]))
out["balance_165W"] = np.array([165.0 * mp.T_ON * mp.CYCLES, heat_out, stored])

# online time: the network against the reference model, on this runtime
timing = {}
for k in (1, 10, 100):
    ps = np.linspace(101.0, 199.0, k)
    timing[f"net_{k}"] = middle(lambda: extremes(centre_history(ps)))
    timing[f"ref_{k}"] = middle(lambda: [mp.reference(p) for p in ps], reps=3 if k == 100 else 5)
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP8.2A_net.pt"))   # the trained weights, beside the report
print(f"online, one new power: network {timing['net_1']*1e3:.2f} ms, reference {timing['ref_1']*1e3:.1f} ms "
      f"({timing['ref_1']/timing['net_1']:.0f} times)")
print(f"energy at 165 W: in {out['balance_165W'][0]:.0f} J, fins {heat_out:.0f} J, stored {stored:.0f} J")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's time `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eighteen questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP8.2A_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): worst {REF['worst_ref_K']:.2f} K, {REF['t_ref_s']*1e3:.1f} ms per new power")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* the worst peak error and the worst trough error at each test power (K);
* the largest error of the field at 600 s at 165 W (K);
* the network's online time for one new power, the reference model's on the same runtime (both in `R`), and their ratio;
* the offline training time.

Then say in one line each whether the three requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>45}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP8.2A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the centre's history against the ground truth at the three test powers, with the error; the error of the field at 600 s; the online times of the network and the reference model for 1, 10 and 100 powers (`R["net_1"]`, `R["ref_10"]` and so on); and the swing of the tenth cycle against the power (`R["centre_sweep"]`, question 16).

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eighteen questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 10 are about your network and its result; 11 to 17 ask you to use it; 18 concludes. The next cell writes `MP8.2A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (with any features you computed from them), layers, neurons and number of weights. Why this size?": """
""",
    "2. Which conditions did you build into the network's form, and which into the loss? Give each loss term's value at the end of training.": """
""",
    "3. How did you handle the twenty switches of the duty cycle? Give the worst peak error with your remedy and without it.": """
""",
    "4. Your training: optimiser(s), steps, learning rates, number of points of each kind, and the offline time on the T4 in seconds.": """
""",
    "5. How did you choose the weights of the loss terms? Show the terms during training in one figure and say which was hardest to bring down.": """
""",
    "6. The peaks and troughs at 115, 165 and 195 W: give the worst error at each power in kelvin. Are all within 0.3 K?": """
""",
    "7. In which cycle is your error largest, and why there?": """
""",
    "8. The field at 600 s at 165 W: where is your largest error, how large, and why there?": """
""",
    "9. The online time to answer a new power: your network against the reference model on the same runtime, in milliseconds, and the ratio. What does each spend its time on?": """
""",
    "10. Ask for 1, 10 and 100 new powers at once: how do the network's and the reference model's times grow? Give the three times of each, and say why they grow so differently.": """
""",
    "11. How much does natural convection lower the tenth peak at 165 W, against the fan alone (S = 0, Ex08's model)? Give both peaks in kelvin.": """
""",
    "12. Your network's energy balance over the window at 165 W: heat in, through the fins, stored, in joules. How far from closing?": """
""",
    "13. Outside the training range: your errors at 90 W and 210 W against the ground truth. How fast does the error grow outside the range?": """
""",
    "14. Single against double precision: train or evaluate in float32. What happens to the accuracy and to the online time?": """
""",
    "15. Train again with another seed. How much do the worst errors of the two runs differ? Is your result reproducible within 0.3 K?": """
""",
    "16. The peak, the trough and the swing of the tenth cycle against the power from 100 to 200 W, from your network. Fit each as a P^n: what are the three n, and what do they say about how strongly natural convection changes the answer across this range of powers?": """
""",
    "17. If the module were on for 30 s instead of 20 s, could your network answer? What would you change to make it answer any duty cycle?": """
""",
    "18. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP8.2A — Thermal Cycling with Natural Convection", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP8.2A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP8.2A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP8.2A_report.md into MP8.2A_report.pdf, with any figure
# saved as MP8.2A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP8.2A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP8.2A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP8.2A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP8.2A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass